# LitGPT vs HF Transformers: Q/K/V Extraction Benchmark (Llama-3.2-1B)

This notebook runs on Colab (T4 GPU) and compares **HF Transformers** vs **LitGPT** for:
- first 100 samples from Alpaca
- inference with `meta-llama/Llama-3.2-1B-Instruct`
- extraction of Q, K, V at the same attention layer index
- saving Q/K/V tensors to one file per framework
- timing and throughput comparison

Notes:
- You need a Hugging Face token with access to Llama 3.2 weights.
- This benchmark measures prompt forward pass + Q/K/V extraction (not autoregressive generation loops).

In [4]:
!pip install -q litgpt transformers datasets torch huggingface_hub sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 121.6 MB/s eta 0:00:00


In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

# ====== User-configurable ======
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
NUM_SAMPLES = 100
LAYER_IDX = 5              # keep identical across both frameworks
MAX_INPUT_TOKENS = 128     # reduce if you hit memory issues
OUT_DIR = Path("qkv_outputs")
STORE_DTYPE = torch.float16
# ===============================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.set_grad_enabled(False)

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        if HF_TOKEN:
            print("HF token loaded from Colab Secrets.")
    except Exception:
        pass

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Enter your Hugging Face token (hf_...): ").strip()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF login: OK")
else:
    raise ValueError("HF token is required for this model. Set Colab secret HF_TOKEN or enter it when prompted.")

Device: cuda
GPU: Tesla T4
HF token loaded from Colab Secrets.
HF login: OK


In [2]:
alpaca = load_dataset("tatsu-lab/alpaca", split=f"train[:{NUM_SAMPLES}]")

def build_alpaca_prompt(row):
    instruction = row.get("instruction", "").strip()
    inp = row.get("input", "").strip()
    if inp:
        return f"Instruction:\n{instruction}\n\nInput:\n{inp}\n\nResponse:\n"
    return f"Instruction:\n{instruction}\n\nResponse:\n"

prompts = [build_alpaca_prompt(r) for r in alpaca]
print("Loaded prompts:", len(prompts))
print(prompts[0][:300])

Loaded prompts: 100
Instruction:
Give three tips for staying healthy.

Response:



In [3]:
def summarize_metrics(name, latencies_s, lengths, peak_mem_bytes):
    lat = np.array(latencies_s, dtype=np.float64)
    total_time = float(lat.sum())
    total_tokens = int(np.sum(lengths))
    return {
        "framework": name,
        "samples": int(len(latencies_s)),
        "total_time_s": total_time,
        "mean_latency_ms": float(lat.mean() * 1000.0),
        "p50_latency_ms": float(np.percentile(lat, 50) * 1000.0),
        "p95_latency_ms": float(np.percentile(lat, 95) * 1000.0),
        "prompt_tokens_total": total_tokens,
        "prompt_tokens_per_s": float(total_tokens / total_time) if total_time > 0 else float("nan"),
        "gpu_peak_mem_gb": float(peak_mem_bytes / 1e9) if peak_mem_bytes is not None else float("nan"),
    }

def split_litgpt_qkv(qkv, attn_module):
    # Reconstruct q,k,v exactly as LitGPT does before RoPE is applied
    cfg = attn_module.config
    B, T, _ = qkv.shape
    n_head = int(cfg.n_head)
    n_query_groups = int(getattr(cfg, "n_query_groups", n_head))
    head_size = int(cfg.head_size)

    q_per_kv = n_head // n_query_groups
    total_qkv = q_per_kv + 2

    qkv = qkv.view(B, T, n_query_groups, total_qkv, head_size)
    qkv = qkv.permute(0, 2, 3, 1, 4)
    q, k, v = qkv.split((q_per_kv, 1, 1), dim=2)

    if n_query_groups != n_head:
        q = q.expand(B, n_query_groups, q_per_kv, T, head_size)
        k = k.expand(B, n_query_groups, q_per_kv, T, head_size)
        v = v.expand(B, n_query_groups, q_per_kv, T, head_size)

    q = q.reshape(B, -1, T, head_size).permute(0, 2, 1, 3).reshape(B, T, -1)
    k = k.reshape(B, -1, T, head_size).permute(0, 2, 1, 3).reshape(B, T, -1)
    v = v.reshape(B, -1, T, head_size).permute(0, 2, 1, 3).reshape(B, T, -1)
    return q, k, v

In [4]:
def run_hf(prompts):
    dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE != "cuda":
        model.to(DEVICE)
    model.eval()

    if not (0 <= LAYER_IDX < len(model.model.layers)):
        raise ValueError(f"LAYER_IDX={LAYER_IDX} out of range for HF model ({len(model.model.layers)} layers)")

    attn = model.model.layers[LAYER_IDX].self_attn
    num_heads = int(model.config.num_attention_heads)
    num_kv_heads = int(getattr(model.config, "num_key_value_heads", num_heads))
    head_dim = int(getattr(model.config, "head_dim", model.config.hidden_size // num_heads))
    hidden_size = num_heads * head_dim

    def expand_kv_to_q_heads(x):
        # HF returns grouped K/V for GQA models; expand to query-head width for parity with LitGPT split
        b, t, _ = x.shape
        x = x.view(b, t, num_kv_heads, head_dim)
        if num_kv_heads != num_heads:
            repeat = num_heads // num_kv_heads
            x = x.repeat_interleave(repeat, dim=2)
        return x.reshape(b, t, hidden_size)

    cache = {}

    def mk_hook(name):
        def _hook(_module, _inp, out):
            cache[name] = out.detach()
        return _hook

    def post_attn_pre_hook(_module, inp):
        # Input to o_proj is post-attention output (heads merged), before output projection.
        x = inp[0] if isinstance(inp, (tuple, list)) else inp
        cache["post_attn"] = x.detach()

    handles = [
        attn.q_proj.register_forward_hook(mk_hook("q")),
        attn.k_proj.register_forward_hook(mk_hook("k")),
        attn.v_proj.register_forward_hook(mk_hook("v")),
    ]

    o_proj = getattr(attn, "o_proj", None) or getattr(attn, "out_proj", None)
    if o_proj is None:
        raise RuntimeError("Unable to locate HF attention output projection module.")
    handles.append(o_proj.register_forward_pre_hook(post_attn_pre_hook))

    q_store = torch.zeros((NUM_SAMPLES, MAX_INPUT_TOKENS, hidden_size), dtype=STORE_DTYPE)
    k_store = torch.zeros_like(q_store)
    v_store = torch.zeros_like(q_store)
    post_attn_store = torch.zeros_like(q_store)
    lengths = []
    latencies = []

    model_device = next(model.parameters()).device

    warm = tokenizer(prompts[0], return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(model_device)
    with torch.inference_mode():
        _ = model(**warm, use_cache=False)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    for i, p in enumerate(prompts):
        cache.clear()
        inputs = tokenizer(p, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(model_device)

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            _ = model(**inputs, use_cache=False)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)

        q = cache["q"][0].to(STORE_DTYPE).cpu()
        k = expand_kv_to_q_heads(cache["k"])[0].to(STORE_DTYPE).cpu()
        v = expand_kv_to_q_heads(cache["v"])[0].to(STORE_DTYPE).cpu()
        post_attn = cache["post_attn"][0].to(STORE_DTYPE).cpu()

        if (
            q.shape[1] != hidden_size
            or k.shape[1] != hidden_size
            or v.shape[1] != hidden_size
            or post_attn.shape[1] != hidden_size
        ):
            raise ValueError(
                f"Unexpected tensor widths: q={q.shape}, k={k.shape}, v={v.shape}, post_attn={post_attn.shape}, expected hidden_size={hidden_size}"
            )

        L = min(q.shape[0], k.shape[0], v.shape[0], post_attn.shape[0], MAX_INPUT_TOKENS)
        lengths.append(int(L))
        q_store[i, :L] = q[:L]
        k_store[i, :L] = k[:L]
        v_store[i, :L] = v[:L]
        post_attn_store[i, :L] = post_attn[:L]

    for h in handles:
        h.remove()

    peak_mem = torch.cuda.max_memory_allocated() if DEVICE == "cuda" else None
    out_file = OUT_DIR / f"hf_qkv_layer{LAYER_IDX}.pt"
    torch.save({
        "framework": "hf_transformers",
        "model_id": MODEL_ID,
        "layer_idx": LAYER_IDX,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "kv_expanded_to_num_heads": True,
        "post_attn_definition": "input_to_attention_output_projection",
        "lengths": torch.tensor(lengths, dtype=torch.int32),
        "q": q_store,
        "k": k_store,
        "v": v_store,
        "post_attn": post_attn_store,
    }, out_file)

    metrics = summarize_metrics("hf_transformers", latencies, lengths, peak_mem)
    metrics["qkv_file"] = str(out_file)
    return metrics


In [5]:
from litgpt import LLM
import huggingface_hub.constants
import inspect
if not hasattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER"):
    setattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER", False)
def run_litgpt(prompts):

    precision = "bf16-true" if DEVICE == "cuda" else "32-true"

    sig = inspect.signature(LLM.load)
    load_kwargs = {}
    if "model" in sig.parameters:
        load_kwargs["model"] = MODEL_ID
    if "distribute" in sig.parameters:
        load_kwargs["distribute"] = "auto"
    if "access_token" in sig.parameters and HF_TOKEN:
        load_kwargs["access_token"] = HF_TOKEN
    if "token" in sig.parameters and HF_TOKEN:
        load_kwargs["token"] = HF_TOKEN

    try:
        llm = LLM.load(**load_kwargs)
    except TypeError:
        # Fallback for older/newer LitGPT signatures
        llm = LLM.load(MODEL_ID)

    if getattr(llm, "model", None) is None:
        if hasattr(llm, "distribute"):
            llm.distribute(
                accelerator="cuda" if DEVICE == "cuda" else "cpu",
                devices=1,
                precision=precision,
            )
        else:
            raise RuntimeError("LitGPT returned model=None and has no distribute() method.")

    if getattr(llm, "model", None) is None:
        raise RuntimeError("LitGPT model is still None after distribute().")

    llm.model.eval()

    if hasattr(llm.model, "transformer") and hasattr(llm.model.transformer, "h"):
        blocks = llm.model.transformer.h
    elif hasattr(llm.model, "layers"):
        blocks = llm.model.layers
    else:
        raise RuntimeError("Unable to locate transformer blocks in LitGPT model")

    if not (0 <= LAYER_IDX < len(blocks)):
        raise ValueError(f"LAYER_IDX={LAYER_IDX} out of range for LitGPT model ({len(blocks)} layers)")

    block = blocks[LAYER_IDX]
    attn = getattr(block, "attn", None) or getattr(block, "self_attn", None)
    if attn is None:
        raise RuntimeError("Unable to locate attention module in LitGPT block")

    qkv_proj = getattr(attn, "attn", None) or getattr(attn, "qkv", None)
    if qkv_proj is None:
        raise RuntimeError("Unable to locate LitGPT qkv projection module for hook")

    out_proj = getattr(attn, "proj", None) or getattr(attn, "o_proj", None) or getattr(attn, "out_proj", None)
    if out_proj is None:
        raise RuntimeError("Unable to locate LitGPT attention output projection module.")

    cache = {}

    def qkv_hook(_module, _inp, out):
        if isinstance(out, (tuple, list)):
            out = out[0]
        cache["qkv"] = out.detach()

    def post_attn_pre_hook(_module, inp):
        # Input to projection is post-attention output (heads merged), before output projection.
        x = inp[0] if isinstance(inp, (tuple, list)) else inp
        cache["post_attn"] = x.detach()

    handles = [
        qkv_proj.register_forward_hook(qkv_hook),
        out_proj.register_forward_pre_hook(post_attn_pre_hook),
    ]

    cfg = llm.model.config
    n_embd = getattr(cfg, "n_embd", None)
    if n_embd is None:
        n_embd = getattr(cfg, "hidden_size", None)
    if n_embd is None:
        n_embd = int(cfg.n_head) * int(cfg.head_size)
    hidden_size = int(n_embd)

    q_store = torch.zeros((NUM_SAMPLES, MAX_INPUT_TOKENS, hidden_size), dtype=STORE_DTYPE)
    k_store = torch.zeros_like(q_store)
    v_store = torch.zeros_like(q_store)
    post_attn_store = torch.zeros_like(q_store)
    lengths = []
    latencies = []

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_device = next(llm.model.parameters()).device

    def encode_prompt(text):
        ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)["input_ids"][0]
        return ids.to(model_device)

    warm_ids = encode_prompt(prompts[0]).unsqueeze(0)
    with torch.inference_mode():
        _ = llm.model(warm_ids)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    class _Wrap:
        pass
    attn_for_split = attn
    if not hasattr(attn_for_split, "config"):
        w = _Wrap()
        w.config = cfg
        attn_for_split = w

    for i, p in enumerate(prompts):
        cache.clear()
        ids = encode_prompt(p)
        x = ids.unsqueeze(0)

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            _ = llm.model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)

        if "qkv" not in cache:
            raise RuntimeError("LitGPT hook did not capture qkv output")
        if "post_attn" not in cache:
            raise RuntimeError("LitGPT hook did not capture post-attention output")

        qkv = cache["qkv"]
        q, k, v = split_litgpt_qkv(qkv, attn_for_split)
        q = q[0].to(STORE_DTYPE).cpu()
        k = k[0].to(STORE_DTYPE).cpu()
        v = v[0].to(STORE_DTYPE).cpu()
        post_attn = cache["post_attn"][0].to(STORE_DTYPE).cpu()

        if (
            q.shape[1] != hidden_size
            or k.shape[1] != hidden_size
            or v.shape[1] != hidden_size
            or post_attn.shape[1] != hidden_size
        ):
            raise ValueError(
                f"Unexpected LitGPT widths: q={q.shape}, k={k.shape}, v={v.shape}, post_attn={post_attn.shape}, expected hidden_size={hidden_size}"
            )

        L = min(q.shape[0], k.shape[0], v.shape[0], post_attn.shape[0], MAX_INPUT_TOKENS)
        lengths.append(int(L))
        q_store[i, :L] = q[:L]
        k_store[i, :L] = k[:L]
        v_store[i, :L] = v[:L]
        post_attn_store[i, :L] = post_attn[:L]

    for h in handles:
        h.remove()

    peak_mem = torch.cuda.max_memory_allocated() if DEVICE == "cuda" else None
    out_file = OUT_DIR / f"litgpt_qkv_layer{LAYER_IDX}.pt"
    torch.save({
        "framework": "litgpt",
        "model_id": MODEL_ID,
        "layer_idx": LAYER_IDX,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "post_attn_definition": "input_to_attention_output_projection",
        "lengths": torch.tensor(lengths, dtype=torch.int32),
        "q": q_store,
        "k": k_store,
        "v": v_store,
        "post_attn": post_attn_store,
    }, out_file)

    metrics = summarize_metrics("litgpt", latencies, lengths, peak_mem)
    metrics["qkv_file"] = str(out_file)
    return metrics


In [ ]:
# !pip uninstall -y transformers
# !pip install transformers>=4.43.0

In [6]:
import transformers
print(transformers.__version__)
# Should be >= 4.43.0

4.57.6


In [7]:
hf_metrics = run_hf(prompts)
litgpt_metrics = run_litgpt(prompts)

comparison = pd.DataFrame([hf_metrics, litgpt_metrics])
comparison = comparison[[
    "framework",
    "samples",
    "total_time_s",
    "mean_latency_ms",
    "p50_latency_ms",
    "p95_latency_ms",
    "prompt_tokens_total",
    "prompt_tokens_per_s",
    "gpu_peak_mem_gb",
    "qkv_file",
]]
comparison

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Converting checkpoint files to LitGPT format.
{'checkpoint_dir': PosixPath('checkpoints/meta-llama/Llama-3.2-1B-Instruct'),
 'debug_mode': False,
 'dtype': None,
 'model_name': None}


Loading weights: model.safetensors: 100%|██████████| 00:13<00:00,  7.25it/s


Saving converted checkpoint to checkpoints/meta-llama/Llama-3.2-1B-Instruct


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


,framework,samples,total_time_s,mean_latency_ms,p50_latency_ms,p95_latency_ms,prompt_tokens_total,prompt_tokens_per_s,gpu_peak_mem_gb,qkv_file
0,hf_transformers,100,7.907703,79.077030,77.424258,84.654064,2214,279.980168,2.516526,qkv_outputs/hf_qkv_layer5.pt
1,litgpt,100,7.273446,72.734456,70.858248,80.282750,2214,304.394936,3.120864,qkv_outputs/litgpt_qkv_layer5.pt


In [8]:
faster = comparison.sort_values("total_time_s").iloc[0]
print("Faster (this run):", faster["framework"])
print("\nSaved files:")
for p in comparison["qkv_file"].tolist():
    print(" -", p)

print("\nTo load a saved tensor file later:")
print("obj = torch.load('<path>.pt', map_location='cpu')")
print("obj.keys() ->", ["framework", "model_id", "layer_idx", "max_input_tokens", "lengths", "q", "k", "v", "post_attn"])


Faster (this run): litgpt

Saved files:
 - qkv_outputs/hf_qkv_layer5.pt
 - qkv_outputs/litgpt_qkv_layer5.pt

To load a saved tensor file later:
obj = torch.load('<path>.pt', map_location='cpu')
obj.keys() -> ['framework', 'model_id', 'layer_idx', 'max_input_tokens', 'lengths', 'q', 'k', 'v', 'post_attn']


In [12]:
import torch
import pandas as pd

hf = torch.load("qkv_outputs/hf_qkv_layer5.pt", map_location="cpu")
lg = torch.load("qkv_outputs/litgpt_qkv_layer5.pt", map_location="cpu")

print("HF keys:", hf.keys())
print("LitGPT keys:", lg.keys())
print("HF q shape:", tuple(hf["q"].shape), "lengths shape:", tuple(hf["lengths"].shape))
print("LitGPT q shape:", tuple(lg["q"].shape))


HF keys: dict_keys(['framework', 'model_id', 'layer_idx', 'max_input_tokens', 'kv_expanded_to_num_heads', 'post_attn_definition', 'lengths', 'q', 'k', 'v', 'post_attn'])
LitGPT keys: dict_keys(['framework', 'model_id', 'layer_idx', 'max_input_tokens', 'post_attn_definition', 'lengths', 'q', 'k', 'v', 'post_attn'])
HF q shape: (100, 128, 2048) lengths shape: (100,)
LitGPT q shape: (100, 128, 2048)


In [11]:
# Inspect actual values for one sample/token
sample_idx = 0
token_idx = 0  # must be < lengths[sample_idx]

L = int(hf["lengths"][sample_idx])
print("valid length:", L)

for name in ["q", "k", "v", "post_attn"]:
    h = hf[name][sample_idx, token_idx, :12]
    l = lg[name][sample_idx, token_idx, :12]
    print(f"\n{name} first 12 dims")
    print("HF    :", h.tolist())
    print("LitGPT:", l.tolist())
    print("mean abs diff:", (h.float() - l.float()).abs().mean().item())


valid length: 12

q first 12 dims
HF    : [-0.11328125, 0.049072265625, 0.0186767578125, -0.035400390625, -0.001800537109375, -0.1396484375, -0.109375, -0.1103515625, -0.0478515625, -0.072265625, 0.039794921875, 0.37890625]
LitGPT: [-0.11328125, 0.04931640625, 0.0191650390625, -0.034912109375, -0.00188446044921875, -0.138671875, -0.10888671875, -0.11083984375, -0.048095703125, -0.07177734375, 0.039794921875, 0.37890625]
mean abs diff: 0.00033251443528570235

k first 12 dims
HF    : [0.0034027099609375, 0.0016021728515625, 6.771087646484375e-05, 0.003753662109375, -0.002471923828125, -0.0014495849609375, 0.00028228759765625, -0.0001583099365234375, -0.0050048828125, -0.001983642578125, -0.002899169921875, 0.0006256103515625]
LitGPT: [-0.279296875, 0.1435546875, -0.11572265625, 0.12890625, -0.341796875, -0.365234375, 0.5546875, 0.07421875, -0.181640625, -0.337890625, 0.36328125, -0.59765625]
mean abs diff: 0.28954100608825684

v first 12 dims
HF    : [0.00677490234375, 0.01080322265625, 